# 03: Model Değerlendirme Fonksiyonları (Kişi 3)

**Proje:** Framingham Kalp Çalışması verisi ile inme (stroke) tahmini

**Bu notebook'ta yapılanlar:**
1. İnme tahmini için hangi metriklerin neden kullanıldığı
2. Model değerlendirme yönteminin belirlenmesi: çapraz doğrulama, eşik seçimi, test seti
3. Ekibin ortak kullanacağı değerlendirme fonksiyonlarının yazılması (`src/metrics.py`)
4. Fonksiyonların model eğitimi sırasında ve sonrasında nasıl çağrılacağı

## 1. Kurulum

Ek kurulum gerekmez: fonksiyonlar yalnızca `numpy`, `pandas`, `matplotlib` ve `scikit-learn` kullanır (Colab'da hazır gelir). SMOTE'lu pipeline kullanılacaksa `imbalanced-learn` gerekir, o da Colab'da yüklüdür. Colab'da `config.py` ve `metrics.py` dosyalarını (pipeline için `preprocessing.py`) sol menüdeki klasör simgesinden yükleyin.

## 2. Hangi metrikler, neden?

Veride inme geçiren kişiler azınlıktadır. Bu yüzden **accuracy (doğruluk) yanıltıcıdır**: örneğin inme oranı %10 olsaydı, herkese "inme yok" diyen ve hiçbir vakayı yakalamayan bir model %90 doğruluk alırdı. Modeller bu nedenle aşağıdaki metriklerle değerlendirilir.

| Metrik | Soru | İnme açısından anlamı |
|---|---|---|
| **Recall** (duyarlılık) | Gerçek inme vakalarının yüzde kaçını yakaladık? | En önemli metrik: kaçırılan her vaka, uyarılmamış bir hasta demektir |
| **Precision** (kesinlik) | "İnme riski var" dediklerimizin yüzde kaçı gerçekten inme geçirdi? | Düşükse çok yanlış alarm var: gereksiz tetkik ve endişe |
| **Specificity** (özgüllük) | İnme geçirmeyenlerin yüzde kaçına doğru "yok" dedik? | Sağlıklı kişileri gereksiz yere riskli saymama oranı |
| **F1** | Recall ve precision'ın harmonik ortalaması | İkisini tek sayıda dengeler |
| **ROC-AUC** | Model inme geçirecek birini geçirmeyecek birinden daha riskli sıralıyor mu? | Eşikten bağımsız, genel ayırt etme gücü |
| **PR-AUC** | Tüm eşiklerde recall ve precision birlikte ne kadar iyi? | Eşikten bağımsız, azınlık sınıfa odaklı |
| Accuracy | Tüm tahminlerin yüzde kaçı doğru? | Yalnızca karşılaştırma için raporlanır, model seçiminde kullanılmaz |

### Neden ana metrik PR-AUC?

- **Recall tek başına seçim metriği olamaz:** herkese "inme var" diyen bir model recall = 1 alır. Recall, precision ile birlikte okunmalıdır.
- **ROC-AUC dengesiz veride iyimserdir:** yanlış alarm oranını, sayıca çok fazla olan sağlıklı kişilere bölerek hesaplar. Yüzlerce yanlış alarm bile bu oranı az değiştirir, eğri iyi görünür.
- **PR-AUC** doğrudan azınlık sınıfa (inme) bakar: yanlış alarmlar precision'ı hemen düşürür. Rastgele bir modelin PR-AUC'si inme oranına eşittir, bu da kıyas için net bir taban verir.

Bu yüzden hiperparametre aramasında en iyi model **PR-AUC** ile seçilir (`MAIN_METRIC`), ardından karar eşiği istenen recall'a göre ayarlanır.

### Değerlendirme yöntemi

- **Çapraz doğrulama (5 katlı, katmanlı, sabit seed):** model seçimi ve hiperparametre araması yalnızca eğitim verisi üzerinde yapılır. Her kat bir kez doğrulama verisi olur, skorlar ortalama ± standart sapma olarak raporlanır. Veri küçük olduğu için standart sapma önemlidir: iki model arasındaki fark bu sapmadan küçükse, aralarında gerçek bir fark olduğu söylenemez.
- **Eğitim - CV farkı:** her katta eğitim skoru da hesaplanır. Eğitim skoru doğrulama skorundan çok yüksekse model eğitim verisini ezberlemiştir (overfitting).
- **Eşik seçimi:** modeller varsayılan olarak olasılık ≥ 0.5 ise "inme var" der. Dengesiz veride bu eşik çok vakayı kaçırır. Eşik, CV tahminleri üzerinde hedef recall'u (varsayılan %80, `TARGET_RECALL`) en az yanlış alarmla sağlayacak şekilde seçilir.
- **Test seti en sona kadar kullanılmaz:** model ve eşik seçimi bittikten sonra final model test setinde **bir kez** değerlendirilir. Test setine bakarak yapılan her seçim, test skorunu yapay olarak iyileştirir.

## 3. Ortak değerlendirme fonksiyonu

Fonksiyonlar `src/metrics.py` dosyasındadır, ortak sabitler (`RANDOM_STATE`, `CV_FOLDS`, `MAIN_METRIC`, `TARGET_RECALL`, `RESULTS_FILE`) `src/config.py`'de. **Ekipteki herkes** bunları kendi notebook'una import edip kullanır:

```python
from metrics import cv_evaluate
result = cv_evaluate(model, X_train, y_train, model_name="SVM")
```

**Ekibe notlar:**
- Model düz bir model ya da pipeline olabilir (ör. `build_pipeline()` çıktısı). **SMOTE mutlaka pipeline içinde olmalı**, SMOTE'lanmış veri dosyaya kaydedilip onunla CV yapılmamalı: aksi halde sentetik örnekler doğrulama katına sızar ve recall gerçekte olduğundan yüksek çıkar.
- Fonksiyonlar modelden bağımsızdır. Model `predict_proba` desteklemiyorsa (ör. `SVC(probability=False)`) `decision_function` kullanılır.
- Herkes aynı CV katlarını (`shared_cv()`) ve aynı metrikleri (`SCORING`) kullanır, böylece modellerin sonuçları birbiriyle karşılaştırılabilir.
- Her `cv_evaluate` çağrısı `results.csv` dosyasına bir satır ekler. Karşılaştırma tablosu bu dosyadan çıkar.
- Rastgelelik içeren her adımda (`SMOTE`, `RandomForestClassifier`, `SVC`...) `random_state=RANDOM_STATE` verilmelidir. Aksi halde her çalıştırmada farklı sonuç çıkar.

In [ ]:
import sys
from pathlib import Path

# Repo: ../src. Colab: the src/*.py files uploaded to the working directory.
SRC_DIR = next((p for p in (Path("../src"), Path("src"), Path(".")) if (p / "config.py").exists()), None)
if SRC_DIR is None:
    raise FileNotFoundError("config.py bulunamadı. Colab'da src klasöründeki .py dosyalarını sol menüden yükleyin.")
sys.path.insert(0, str(SRC_DIR.resolve()))

from config import MAIN_METRIC, RANDOM_STATE, TARGET_RECALL
from metrics import (SCORING, compare_models, cv_evaluate, learning_curve_analysis, plot_curves,
                     shared_cv, test_evaluate, threshold_analysis)

## 4. Fonksiyonların içindeki adımlar

| Fonksiyon | Ne yapar | Ne döndürür |
|---|---|---|
| `shared_cv()`, `SCORING`, `MAIN_METRIC` | Ortak CV katları ve metrikler. Hiperparametre aramasına verilir | `StratifiedKFold`, metrik sözlüğü, `"pr_auc"` |
| `cv_evaluate(model, X_train, y_train, model_name)` | Modeli 5 katta sıfırdan eğitir, her katta eğitim ve doğrulama skorlarını hesaplar, sonucu `results.csv`'ye ekler | Metrik özeti (`summary`: eğitim ort., CV ort., CV std, fark), kat sonuçları (`folds`), CV tahminleri (`oof_score`) |
| `threshold_analysis(y, oof_score, target_recall)` | Eşiğe göre yakalanan vaka, kaçırılan vaka, yanlış alarm tablosu; hedef recall'u sağlayan en yüksek eşiği seçer | `(table, chosen_threshold)` |
| `learning_curve_analysis(model, X_train, y_train)` | Eğitim verisi büyüdükçe eğitim ve CV skorunu çizer | Boyut / skor tablosu |
| `test_evaluate(model, X_test, y_test, threshold)` | Final modeli test setinde bir kez değerlendirir: karmaşıklık matrisi, ROC ve PR eğrileri, CV ile test farkı | Test metrikleri, karmaşıklık matrisi, test skorları |
| `compare_models()` | `results.csv`'deki tüm modelleri tablo ve grafiklerle karşılaştırır | Model × metrik tablosu |
| `plot_curves({name: (y, score)})` | Birden fazla modelin ROC ve PR eğrilerini aynı grafikte çizer | (grafik) |

**OOF (out-of-fold) tahmini nedir?** CV'de her eğitim satırı tam olarak bir kez doğrulama katına düşer. O satırın skoru, onu hiç görmemiş modelden gelir. Böylece test setine dokunmadan, tüm eğitim verisi için "görülmemiş veri" tahmini elde edilir. Eşik bu tahminlerle seçilir.

## 5. Ekip için kullanım

Her ekip üyesi kendi model notebook'unda şu adımları uygular.

**1) Hiperparametre araması: ortak katlar ve ortak metriklerle**

```python
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC
from config import MAIN_METRIC
from metrics import SCORING, shared_cv
from preprocessing import build_pipeline, load_data

X_train, X_test, y_train, y_test = load_data()
model = build_pipeline(SVC(probability=True), balancing="smote")
search = GridSearchCV(model, {"model__C": [0.1, 1, 10]}, cv=shared_cv(),
                      scoring=SCORING, refit=MAIN_METRIC, n_jobs=-1)
search.fit(X_train, y_train)
best = search.best_estimator_
```

**2) En iyi modelin CV değerlendirmesi, eşik ve overfitting kontrolü**

```python
from metrics import cv_evaluate, threshold_analysis, learning_curve_analysis

result = cv_evaluate(best, X_train, y_train, model_name="SVM",
                     note="SMOTE + SVM, C araması")
table, threshold = threshold_analysis(result["y"], result["oof_score"], model_name="SVM")
learning_curve_analysis(best, X_train, y_train, model_name="SVM")
```

**3) Final: test seti (yalnızca en sonda, bir kez)**

```python
from metrics import test_evaluate

best.fit(X_train, y_train)                     # tüm eğitim verisiyle
test = test_evaluate(best, X_test, y_test, threshold=threshold,
                     model_name="SVM", cv_result=result)
```

**4) Tüm modellerin karşılaştırılması** (herkes `results.csv`'ye satır ekledikten sonra):

```python
from metrics import compare_models, plot_curves

compare_models("results.csv")
plot_curves({"Logistic Regression": (lr_result["y"], lr_result["oof_score"]),
             "SVM": (svm_result["y"], svm_result["oof_score"]),
             "Random Forest": (rf_result["y"], rf_result["oof_score"])})
```

Aynı eğitilmiş model XAI için de doğrudan kullanılabilir: `explain(best, X_train, X_test, model_name="SVM")` (`src/explain.py`).

## 6. Özet (rapor için)

- Sınıflar dengesiz olduğu için accuracy model seçiminde kullanılmadı. Modeller recall, precision, specificity, F1, ROC-AUC ve PR-AUC ile değerlendirildi.
- İnmeyi kaçırmak yanlış alarmdan daha pahalı olduğu için recall öne çıkarıldı. Ancak recall tek başına yanıltıcı olabildiği için (herkese "inme var" demek recall = 1 verir) en iyi model, azınlık sınıfa odaklı ve eşikten bağımsız olan PR-AUC ile seçildi.
- Tüm modeller aynı 5 katlı katmanlı çapraz doğrulama bölmesiyle değerlendirildi. Skorlar ortalama ± standart sapma olarak raporlandı. Her katta eğitim skoru da hesaplanarak overfitting kontrol edildi.
- SMOTE pipeline içinde, yalnızca eğitim katlarına uygulandı. Böylece doğrulama katları gerçek veri dağılımını korudu.
- Karar eşiği, test setine bakılmadan, CV tahminleri üzerinde hedef recall'u en az yanlış alarmla sağlayacak şekilde seçildi.
- Final modeller test setinde yalnızca bir kez değerlendirildi; CV ve test skorları karşılaştırılarak modelin yeni veriye genellenip genellenmediği incelendi.
- Tüm ekibin kullanabileceği, modelden bağımsız ortak değerlendirme fonksiyonları (`src/metrics.py`) yazıldı.